In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Pipeline exploratoire LFP autour de stimulations :
- lecture des TRC
- fusion des tables d'événements
- exclusion bad channels
- montage bipolaire adjacent
- filtrage
- extraction pre/post
- TFR Morlet
- normalisation baseline
- sauvegarde des résultats

Auteur: à adapter si besoin
"""

from __future__ import annotations

import ast
import json
import math
import re
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy import signal
import mne

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------

@dataclass
class Config:
    # Dossiers
    root_dir: str = "/home/aube/Documents/article_neuronal_stimic/effets_cog/theta_gamma_cog"
    output_dir: str = "/home/aube/Documents/article_neuronal_stimic/effets_cog/theta_gamma_cog/results_morlet_exploratoire"

    # Epoching
    pre_length: float = 5.0                # t_start - 10 -> t_start
    post_length: float = 3.0                # durée post par défaut
    epsilon: float = 0.1                    # marge avant début stim et après fin stim
    use_anchor_start_for_pre: bool = True   # pre = [t_start - pre_length, t_start]
    use_anchor_end_for_post: bool = True    # post = [t_start+duration+epsilon, ...]
    allow_global_baseline: bool = True      # option pour baseline [0, première stim]

    # Filtrage
    do_notch: bool = True
    notch_freqs: Tuple[float, ...] = (50.0, 100.0, 150.0)
    notch_q: float = 30.0
    do_highpass: bool = True
    highpass_hz: float = 0.1

    # Morlet
    fmin: float = 4.0 # descendre à 2 ok si duration post est plus longue que 3 seulement, car ondelette très longue par rapport a l'epoch sinon
    fmax: float = 150.0
    n_freqs: int = 80 #(plante avec 149)
    freq_scale: str = "linear"  # "linear" | "log" | "semilog"
    n_cycles: float = 7.0
    decim: int = 4              # décimation après TFR si besoin
    n_jobs: int = 1
    save_per_channel: bool = True

    # Baseline normalisation
    baseline_mode: str = "trial_pre"  # "trial_pre" | "global_pre_first_stim"
    baseline_stat: str = "median"     # "median" | "mean"
    baseline_metric: str = "logratio" # "logratio" | "percent" | "zscore" | "subtract"
    eps: float = 1e-12

    # Sauvegarde
    save_raw_epochs: bool = False
    save_filtered_signal_preview: bool = False

    # Divers
    verbose: bool = True


CFG = Config()


# ---------------------------------------------------------------------
# OUTILS GÉNÉRAUX
# ---------------------------------------------------------------------

def log(msg: str) -> None:
    if CFG.verbose:
        print(msg, flush=True)


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def make_freqs(fmin: float, fmax: float, n_freqs: int, scale: str = "linear") -> np.ndarray:
    if scale == "linear":
        return np.linspace(fmin, fmax, n_freqs)
    if scale == "log":
        return np.geomspace(fmin, fmax, n_freqs)
    if scale == "semilog":
        # compromis simple : plus dense en bas, plus étalé en haut
        x = np.linspace(0.0, 1.0, n_freqs)
        return fmin * (fmax / fmin) ** x
    raise ValueError(f"freq_scale inconnu: {scale}")


def parse_bad_channels_cell(cell_value) -> List[str]:
    """
    Gère des cellules Excel du type:
    - "Bp1, C2"
    - "Bp1; C2"
    - "[Bp1, C2]" (au cas où)
    - NaN
    """
    if pd.isna(cell_value):
        return []

    s = str(cell_value).strip()
    if not s:
        return []

    # enlève crochets éventuels
    s = s.strip("[](){}")

    # split sur virgule ou point-virgule
    parts = re.split(r"[;,]", s)
    chs = [p.strip() for p in parts if p.strip()]
    return chs


def normalize_channel_name(ch: str) -> str:
    """
    Normalisation légère des labels de canaux.
    On retire espaces, on uniformise l'underscore, etc.
    À adapter si tes TRC ont une convention spécifique.
    """
    s = str(ch).strip()
    s = s.replace(" ", "")
    return s


def parse_shaft_and_contact(ch_name: str) -> Optional[Tuple[str, int]]:
    """
    Extrait:
      - shaft_id = partie lettre / underscore
      - contact = partie numérique finale

    Exemples:
      Bp1   -> ("Bp", 1)
      Bp_1  -> ("Bp", 1)
      A'10  -> ("A'", 10)
      C_12  -> ("C", 12)

    Retourne None si non parseable.
    """
    s = normalize_channel_name(ch_name)
    m = re.match(r"^([A-Za-zÀ-ÿ'_-]+?)(\d+)$", s)
    if not m:
        return None

    shaft = m.group(1).replace("_", "")
    contact = int(m.group(2))
    return shaft, contact


def build_adjacent_bipolar_pairs(channel_names: Sequence[str],
                                 bad_channels: Sequence[str]) -> List[Tuple[str, str, str]]:
    """
    Construit des paires adjacentes présentes seulement:
      1-2, 2-3, 3-4...
    mais PAS 3-6 si 4 et 5 absents.

    Retour:
      [(new_name, ch_a, ch_b), ...]
    """
    bad_set = {normalize_channel_name(ch) for ch in bad_channels}
    good = [normalize_channel_name(ch) for ch in channel_names if normalize_channel_name(ch) not in bad_set]

    grouped: Dict[str, List[Tuple[int, str]]] = {}
    for ch in good:
        parsed = parse_shaft_and_contact(ch)
        if parsed is None:
            continue
        shaft, num = parsed
        grouped.setdefault(shaft, []).append((num, ch))

    pairs: List[Tuple[str, str, str]] = []
    for shaft, vals in grouped.items():
        vals = sorted(vals, key=lambda x: x[0])
        num_to_ch = {num: ch for num, ch in vals}
        nums = sorted(num_to_ch)

        for n in nums:
            if (n + 1) in num_to_ch:
                ch1 = num_to_ch[n]
                ch2 = num_to_ch[n + 1]
                new_name = f"{shaft}{n}-{shaft}{n+1}"
                pairs.append((new_name, ch1, ch2))

    return pairs


def classify_group_and_cog_labels(cog_value) -> Tuple[str, List[str]]:
    """
    cog peut être:
    - NaN
    - 'controle'
    - 'negatif'
    - "['souvenir', 'emotion']"
    - "souvenir, emotion"
    - etc.

    Retour:
      group_label in {"cog+", "controle", "negatif", "unknown"}
      cog_labels  = liste des labels cognitifs exacts si cog+
    """
    if pd.isna(cog_value):
        return "unknown", []

    s = str(cog_value).strip()
    if not s:
        return "unknown", []

    labels: List[str] = []

    # Essai literal_eval
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple, set)):
            labels = [str(x).strip() for x in parsed if str(x).strip()]
        else:
            labels = [str(parsed).strip()]
    except Exception:
        # split rudimentaire
        parts = re.split(r"[;,/]", s)
        labels = [p.strip().strip("'\"") for p in parts if p.strip()]

    # fallback si rien
    if not labels:
        labels = [s]

    labels_low = [x.lower() for x in labels]

    has_neg = any(x == "negatif" for x in labels_low)
    has_ctrl = any(x == "controle" for x in labels_low)

    # labels cognitifs = tout ce qui n'est pas controle/negatif
    cog_labels = [lab for lab, low in zip(labels, labels_low) if low not in {"negatif", "controle"}]

    # règle de priorité explicite
    if has_neg and not has_ctrl and not cog_labels:
        return "negatif", []
    if has_ctrl and not has_neg and not cog_labels:
        return "controle", []
    if cog_labels:
        return "cog+", cog_labels

    # cas ambigus
    if has_neg and has_ctrl:
        return "unknown", []
    if has_neg and cog_labels:
        return "unknown", cog_labels
    if has_ctrl and cog_labels:
        return "unknown", cog_labels

    return "unknown", []


# ---------------------------------------------------------------------
# LECTURE ÉVÉNEMENTS / BAD CHANNELS
# ---------------------------------------------------------------------

def list_trc_sessions(root_dir: Path) -> List[str]:
    """
    Cherche les fichiers *.TRC et retourne les noms de session sans suffixe.
    """
    sessions = []
    for fp in root_dir.glob("*.TRC"):
        sessions.append(fp.stem)
    return sorted(set(sessions))


def find_cog_file(root_dir: Path, session: str) -> Path:
    matches = list(root_dir.glob(f"{session}_stim_events_TRC_re-shifted_*COG.txt"))
    if len(matches) == 0:
        raise FileNotFoundError(f"Aucun fichier COG trouvé pour {session}")
    if len(matches) > 1:
        log(f"[WARN] Plusieurs fichiers COG pour {session}, je prends le premier: {matches[0].name}")
    return matches[0]


def find_duration_file(root_dir: Path, session: str) -> Path:
    fp = root_dir / f"{session}_stim_events_TRC_duration.txt"
    if not fp.exists():
        raise FileNotFoundError(f"Fichier duration introuvable: {fp}")
    return fp


def read_cog_file(cog_file: Path) -> pd.DataFrame:
    """
    Format annoncé:
      stim;t_start;duration;lobe;cog
    avec t_start faux
    """
    df = pd.read_csv(cog_file, sep=";", dtype=str)
    expected = {"stim", "t_start", "duration", "lobe", "cog"}
    missing = expected - set(df.columns)
    if missing:
        raise ValueError(f"Colonnes manquantes dans {cog_file.name}: {missing}")

    df = df.rename(columns={
        "stim": "label_stim",
        "cog": "cog"
    })
    return df[["label_stim", "t_start", "duration", "lobe", "cog"]].copy()


def read_duration_file(duration_file: Path) -> pd.DataFrame:
    """
    Table sans noms de colonnes.
    Colonnes attendues:
      label_stim, t_start, duration
    """
    df = pd.read_csv(duration_file, sep=',', engine="python", header=None)
    if df.shape[1] < 3:
        raise ValueError(f"Le fichier {duration_file.name} doit avoir au moins 3 colonnes")
    df = df.iloc[:, :3].copy()
    df.columns = ["label_stim", "t_start", "duration"]

    # conversion robuste
    df["label_stim"] = df["label_stim"].astype(str).str.strip()
    df["t_start"] = pd.to_numeric(df["t_start"], errors="coerce")
    df["duration"] = pd.to_numeric(df["duration"], errors="coerce")
    return df


def merge_event_tables(session: str, cog_df: pd.DataFrame, dur_df: pd.DataFrame) -> pd.DataFrame:
    """
    Fusion stricte par ordre et vérification des labels.
    Les bons t_start/duration viennent de dur_df.
    """
    if len(cog_df) != len(dur_df):
        raise ValueError(
            f"{session}: nombre de stimulations différent entre COG ({len(cog_df)}) et duration ({len(dur_df)})"
        )

    cog_df = cog_df.reset_index(drop=True).copy()
    dur_df = dur_df.reset_index(drop=True).copy()

    lbl1 = cog_df["label_stim"].astype(str).str.strip().tolist()
    lbl2 = dur_df["label_stim"].astype(str).str.strip().tolist()

    if lbl1 != lbl2:
        # tentative par merge sur label si ordre différent
        tmp = pd.merge(
            dur_df,
            cog_df[["label_stim", "lobe", "cog"]],
            on="label_stim",
            how="inner",
            validate="one_to_one",
        )
        if len(tmp) != len(dur_df):
            raise ValueError(f"{session}: impossible d'aligner strictement les tables événements")
        merged = tmp.copy()
    else:
        merged = dur_df.copy()
        merged["lobe"] = cog_df["lobe"].values
        merged["cog"] = cog_df["cog"].values

    groups = merged["cog"].apply(classify_group_and_cog_labels)
    merged["group_label"] = groups.apply(lambda x: x[0])
    merged["cog_labels"] = groups.apply(lambda x: x[1])

    # ordre de stimulation
    merged["stim_index"] = np.arange(len(merged), dtype=int)

    # validation
    merged["t_start"] = pd.to_numeric(merged["t_start"], errors="coerce")
    merged["duration"] = pd.to_numeric(merged["duration"], errors="coerce")
    bad_rows = merged["t_start"].isna() | merged["duration"].isna()
    if bad_rows.any():
        raise ValueError(f"{session}: t_start/duration NaN dans certaines lignes après fusion")

    return merged


def load_bad_channels_table(root_dir: Path) -> pd.DataFrame:
    xlsx = root_dir / "TRC_bad_channels.xlsx"
    if not xlsx.exists():
        raise FileNotFoundError(f"Fichier bad channels introuvable: {xlsx}")
    df = pd.read_excel(xlsx)
    if not {"session", "bad_channels"}.issubset(df.columns):
        raise ValueError("TRC_bad_channels.xlsx doit contenir les colonnes 'session' et 'bad_channels'")
    df["session"] = df["session"].astype(str).str.strip()
    return df


def get_bad_channels_for_session(bad_df: pd.DataFrame, session: str) -> List[str]:
    sub = bad_df.loc[bad_df["session"] == session]
    if len(sub) == 0:
        return []
    vals = []
    for cell in sub["bad_channels"].tolist():
        vals.extend(parse_bad_channels_cell(cell))
    return [normalize_channel_name(x) for x in vals]


# ---------------------------------------------------------------------
# LECTURE TRC
# ---------------------------------------------------------------------

def load_trc_as_mne_raw(trc_path: Path) -> mne.io.BaseRaw:
    """
    Lecture robuste d'un TRC via MicromedTRC, puis construction manuelle
    d'un RawArray MNE en convertissant les données en Volts.

    Hypothèse :
    - les données lues sont en µV (cohérent avec mmtrc._header["chans"][i]["units"] == "μV")
    - on les convertit en V pour MNE
    """
    from micromed_io.trc import MicromedTRC

    mmtrc = MicromedTRC(str(trc_path))

    # Fréquence d'échantillonnage
    sfreq = float(mmtrc._header["s_freq"])

    # Noms de canaux
    chans = mmtrc._header["chans"]
    ch_names = [normalize_channel_name(ch["chan_name"]) for ch in chans]

    # Lecture données
    # Selon les versions de micromed_io, l'accès peut être mmtrc.get_data()
    # ou mmtrc.data. On gère les deux.
    if hasattr(mmtrc, "get_data"):
        data = mmtrc.get_data()
    elif hasattr(mmtrc, "data"):
        data = mmtrc.data
    else:
        raise AttributeError(
            "Impossible de trouver les données dans l'objet MicromedTRC "
            "(ni get_data(), ni attribut data)."
        )

    data = np.asarray(data, dtype=np.float64)
    log(f"[INFO] {trc_path.name}: data shape avant RawArray = {data.shape}")
    log(f"[INFO] {trc_path.name}: sfreq = {sfreq}")

    # On veut une matrice (n_channels, n_times)
    if data.ndim != 2:
        raise ValueError(f"Données TRC de dimension inattendue: shape={data.shape}")

    # Si la forme est (n_times, n_channels), on transpose
    if data.shape[0] != len(ch_names) and data.shape[1] == len(ch_names):
        data = data.T

    if data.shape[0] != len(ch_names):
        raise ValueError(
            f"Incohérence données/canaux pour {trc_path.name}: "
            f"data.shape={data.shape}, n_channels={len(ch_names)}"
        )

    # Vérification informative des unités canal par canal
    units = [str(ch.get("units", "")).strip() for ch in chans]
    unique_units = sorted(set(units))
    log(f"[INFO] {trc_path.name}: unités canal détectées = {unique_units}")

    # Conversion µV -> V pour MNE
    # MNE attend les données en Volts pour les canaux EEG/iEEG.
    data_volts = data * 1e-6

    # Type de canal : intracrânien.
    # Si ta version MNE n'aime pas 'seeg', remplace par 'eeg'.
    ch_types = ["seeg"] * len(ch_names)

    info = mne.create_info(
        ch_names=ch_names,
        sfreq=sfreq,
        ch_types=ch_types,
    )

    raw = mne.io.RawArray(data_volts, info, verbose=False)
    return raw

# ---------------------------------------------------------------------
# PRÉTRAITEMENT
# ---------------------------------------------------------------------

def apply_filters(data: np.ndarray,
                  sfreq: float,
                  do_notch: bool = True,
                  notch_freqs: Sequence[float] = (50, 100, 150),
                  notch_q: float = 30.0,
                  do_highpass: bool = True,
                  highpass_hz: float = 0.1) -> np.ndarray:
    """
    data shape = (n_channels, n_samples)
    Filtrage zero-phase via scipy.signal.
    """
    x = np.asarray(data, dtype=np.float64).copy()

    # Notch successifs
    if do_notch:
        for f0 in notch_freqs:
            if f0 >= sfreq / 2:
                continue
            b, a = signal.iirnotch(w0=f0, Q=notch_q, fs=sfreq)
            x = signal.filtfilt(b, a, x, axis=-1)

    # High-pass Butterworth
    if do_highpass and highpass_hz is not None and highpass_hz > 0:
        sos = signal.butter(N=4, Wn=highpass_hz, btype="highpass", fs=sfreq, output="sos")
        x = signal.sosfiltfilt(sos, x, axis=-1)

    return x


def make_bipolar_data(data: np.ndarray,
                      ch_names: Sequence[str],
                      bipolar_pairs: Sequence[Tuple[str, str, str]]) -> Tuple[np.ndarray, List[str]]:
    """
    data shape = (n_channels, n_samples)
    bipolar_pairs = [(new_name, ch_a, ch_b), ...]
    """
    ch_to_idx = {normalize_channel_name(ch): i for i, ch in enumerate(ch_names)}

    bp_data = []
    bp_names = []

    for new_name, ch_a, ch_b in bipolar_pairs:
        if normalize_channel_name(ch_a) not in ch_to_idx:
            continue
        if normalize_channel_name(ch_b) not in ch_to_idx:
            continue
        i = ch_to_idx[normalize_channel_name(ch_a)]
        j = ch_to_idx[normalize_channel_name(ch_b)]
        bp_data.append(data[i] - data[j])
        bp_names.append(new_name)

    if len(bp_data) == 0:
        return np.empty((0, data.shape[1])), []

    return np.asarray(bp_data, dtype=np.float64), bp_names


# ---------------------------------------------------------------------
# EPOCHING
# ---------------------------------------------------------------------

def compute_trial_windows(row: pd.Series,
                          pre_length: float,
                          post_length: float,
                          epsilon: float) -> Dict[str, float]:
    """
    Retourne:
      pre_start, pre_end, post_start, post_end
    """
    t_start = float(row["t_start"])
    dur = float(row["duration"])
    t_end = t_start + dur

    pre_start = t_start - pre_length - epsilon
    pre_end = t_start - epsilon

    post_start = t_end + epsilon
    post_end = post_start + post_length

    return {
        "pre_start": pre_start,
        "pre_end": pre_end,
        "post_start": post_start,
        "post_end": post_end,
        "stim_start": t_start,
        "stim_end": t_end,
    }


def add_windows_to_trials(trials_df: pd.DataFrame,
                          pre_length: float,
                          post_length: float,
                          epsilon: float) -> pd.DataFrame:
    out = trials_df.copy()
    windows = out.apply(
        compute_trial_windows,
        axis=1,
        pre_length=pre_length,
        post_length=post_length,
        epsilon=epsilon
    )
    wdf = pd.DataFrame(list(windows))
    out = pd.concat([out.reset_index(drop=True), wdf.reset_index(drop=True)], axis=1)
    return out


def keep_trials_fitting_signal(trials_df: pd.DataFrame,
                               signal_duration_s: float) -> pd.DataFrame:
    good = (
        (trials_df["pre_start"] >= 0) &
        (trials_df["post_end"] <= signal_duration_s)
    )
    dropped = (~good).sum()
    if dropped > 0:
        log(f"[INFO] {dropped} stimulations exclues car fenêtres hors signal")
    return trials_df.loc[good].reset_index(drop=True)


def extract_pre_post_epochs(data_bp: np.ndarray,
                            sfreq: float,
                            trials_df: pd.DataFrame,
                            pre_length: float,
                            post_length: float) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Extrait deux tenseurs séparés:
      pre_epochs  shape = (n_trials, n_channels, n_pre_samples)
      post_epochs shape = (n_trials, n_channels, n_post_samples)

    Retourne aussi:
      pre_times, post_times
    """
    n_pre = int(round(pre_length * sfreq))
    n_post = int(round(post_length * sfreq))

    pre_epochs = []
    post_epochs = []

    for _, row in trials_df.iterrows():
        pre_start_idx = int(round(row["pre_start"] * sfreq))
        pre_end_idx = pre_start_idx + n_pre

        post_start_idx = int(round(row["post_start"] * sfreq))
        post_end_idx = post_start_idx + n_post

        pre_ep = data_bp[:, pre_start_idx:pre_end_idx]
        post_ep = data_bp[:, post_start_idx:post_end_idx]

        if pre_ep.shape[1] != n_pre or post_ep.shape[1] != n_post:
            continue

        pre_epochs.append(pre_ep)
        post_epochs.append(post_ep)

    pre_epochs = np.asarray(pre_epochs, dtype=np.float64)
    post_epochs = np.asarray(post_epochs, dtype=np.float64)

    pre_times = np.arange(n_pre) / sfreq - pre_length
    post_times = np.arange(n_post) / sfreq

    return pre_epochs, post_epochs, pre_times, post_times


def build_global_baseline_segment(data_bp: np.ndarray,
                                  sfreq: float,
                                  first_stim_t_start: float) -> Tuple[np.ndarray, np.ndarray]:
    """
    Segment [0, première stim], pour baseline globale optionnelle.
    """
    stop_idx = int(round(first_stim_t_start * sfreq))
    stop_idx = max(stop_idx, 1)
    seg = data_bp[:, :stop_idx]
    times = np.arange(seg.shape[1]) / sfreq
    return seg, times


# ---------------------------------------------------------------------
# MORLET
# ---------------------------------------------------------------------

def compute_morlet_power(epochs: np.ndarray,
                         sfreq: float,
                         freqs: np.ndarray,
                         n_cycles: float | np.ndarray,
                         decim: int = 1,
                         n_jobs: int = 1) -> np.ndarray:
    """
    epochs shape = (n_epochs, n_channels, n_times)
    return shape = (n_epochs, n_channels, n_freqs, n_times_decim)
    """
    power = mne.time_frequency.tfr_array_morlet(
        data=epochs,
        sfreq=sfreq,
        freqs=freqs,
        n_cycles=n_cycles,
        output="power",
        decim=decim,
        n_jobs=n_jobs,
        zero_mean=True,
    )
    return np.asarray(power, dtype=np.float32)


# ---------------------------------------------------------------------
# BASELINE NORMALISATION
# ---------------------------------------------------------------------

def reduce_baseline_stat(x: np.ndarray, stat: str) -> np.ndarray:
    """
    x shape = (..., n_times)
    return shape = (...)
    """
    if stat == "median":
        return np.median(x, axis=-1)
    if stat == "mean":
        return np.mean(x, axis=-1)
    raise ValueError(f"baseline_stat inconnu: {stat}")


def baseline_normalize_trial_pre(power_pre: np.ndarray,
                                 power_post: np.ndarray,
                                 metric: str,
                                 stat: str,
                                 eps: float = 1e-12) -> Tuple[np.ndarray, np.ndarray]:
    """
    power_pre  shape = (n_trials, n_channels, n_freqs, n_pre_times)
    power_post shape = (n_trials, n_channels, n_freqs, n_post_times)

    baseline calculée par essai / canal / fréquence à partir du pre.
    Retour:
      norm_post, baseline_values
    """
    baseline = reduce_baseline_stat(power_pre, stat=stat)  # (n_trials, n_channels, n_freqs)
    b = baseline[..., None]

    if metric == "logratio":
        norm_post = np.log((power_post + eps) / (b + eps))
    elif metric == "percent":
        norm_post = ((power_post - b) / (b + eps)) * 100.0
    elif metric == "zscore":
        mu = np.mean(power_pre, axis=-1)[..., None]
        sd = np.std(power_pre, axis=-1, ddof=0)[..., None]
        norm_post = (power_post - mu) / (sd + eps)
    elif metric == "subtract":
        norm_post = power_post - b
    else:
        raise ValueError(f"metric inconnue: {metric}")

    return norm_post.astype(np.float32), baseline.astype(np.float32)


def baseline_normalize_global(power_post: np.ndarray,
                              power_global_base: np.ndarray,
                              metric: str,
                              stat: str,
                              eps: float = 1e-12) -> Tuple[np.ndarray, np.ndarray]:
    """
    power_post        shape = (n_trials, n_channels, n_freqs, n_post_times)
    power_global_base shape = (n_channels, n_freqs, n_base_times)

    baseline globale par canal / fréquence, dupliquée sur les essais.
    """
    baseline = reduce_baseline_stat(power_global_base, stat=stat)  # (n_channels, n_freqs)
    b = baseline[None, ..., None]  # (1, n_channels, n_freqs, 1)

    if metric == "logratio":
        norm_post = np.log((power_post + eps) / (b + eps))
    elif metric == "percent":
        norm_post = ((power_post - b) / (b + eps)) * 100.0
    elif metric == "zscore":
        mu = np.mean(power_global_base, axis=-1)[None, ..., None]
        sd = np.std(power_global_base, axis=-1, ddof=0)[None, ..., None]
        norm_post = (power_post - mu) / (sd + eps)
    elif metric == "subtract":
        norm_post = power_post - b
    else:
        raise ValueError(f"metric inconnue: {metric}")

    return norm_post.astype(np.float32), baseline.astype(np.float32)


# ---------------------------------------------------------------------
# SAUVEGARDE
# ---------------------------------------------------------------------

def save_session_outputs(session_out: Path,
                         session: str,
                         config: Config,
                         trials_df: pd.DataFrame,
                         raw_ch_names: List[str],
                         bad_channels: List[str],
                         bp_names: List[str],
                         freqs: np.ndarray,
                         pre_times: np.ndarray,
                         post_times: np.ndarray,
                         power_pre: np.ndarray,
                         power_post: np.ndarray,
                         power_post_norm: np.ndarray,
                         baseline_values: np.ndarray,
                         power_global_base: Optional[np.ndarray] = None) -> None:
    ensure_dir(session_out)

    trials_df.to_csv(session_out / f"{session}_trial_table.csv", index=False)

    meta = {
        "session": session,
        "config": asdict(config),
        "n_trials": int(len(trials_df)),
        "n_raw_channels": int(len(raw_ch_names)),
        "n_bad_channels": int(len(bad_channels)),
        "n_bipolar_channels": int(len(bp_names)),
        "raw_ch_names": raw_ch_names,
        "bad_channels": bad_channels,
        "bipolar_names": bp_names,
    }
    with open(session_out / f"{session}_metadata.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    np.save(session_out / f"{session}_freqs.npy", freqs)
    np.save(session_out / f"{session}_pre_times.npy", pre_times)
    np.save(session_out / f"{session}_post_times.npy", post_times)

    np.save(session_out / f"{session}_power_pre.npy", power_pre)
    np.save(session_out / f"{session}_power_post.npy", power_post)
    np.save(session_out / f"{session}_power_post_norm.npy", power_post_norm)
    np.save(session_out / f"{session}_baseline_values.npy", baseline_values)

    if power_global_base is not None:
        np.save(session_out / f"{session}_power_global_base.npy", power_global_base)


# ---------------------------------------------------------------------
# PIPELINE SESSION
# ---------------------------------------------------------------------

def process_one_session(session: str,
                        root_dir: Path,
                        out_dir: Path,
                        bad_df: pd.DataFrame,
                        cfg: Config) -> None:
    log(f"\n=== Session {session} ===")

    trc_path = root_dir / f"{session}.TRC"
    cog_file = find_cog_file(root_dir, session)
    dur_file = find_duration_file(root_dir, session)

    # 1) événements
    cog_df = read_cog_file(cog_file)
    dur_df = read_duration_file(dur_file)
    trials_df = merge_event_tables(session, cog_df, dur_df)
    trials_df = add_windows_to_trials(
        trials_df,
        pre_length=cfg.pre_length,
        post_length=cfg.post_length,
        epsilon=cfg.epsilon,
    )

    # 2) TRC
    raw = load_trc_as_mne_raw(trc_path)
    log(f"[INFO] {trc_path.name}: RawArray créé, shape = {raw.get_data().shape}")

    sfreq = float(raw.info["sfreq"])
    raw_ch_names = [normalize_channel_name(ch) for ch in raw.ch_names]
    data = raw.get_data()  # shape (n_channels, n_samples)
    signal_duration_s = data.shape[1] / sfreq

    # 3) restreindre aux essais qui tiennent dans le signal
    trials_df = keep_trials_fitting_signal(trials_df, signal_duration_s)
    if len(trials_df) == 0:
        log(f"[WARN] {session}: aucun essai exploitable après contrôle des fenêtres")
        return

    # 4) bad channels
    bad_channels = get_bad_channels_for_session(bad_df, session)

    # 5) montage bipolaire
    bipolar_pairs = build_adjacent_bipolar_pairs(raw_ch_names, bad_channels)
    if len(bipolar_pairs) == 0:
        log(f"[WARN] {session}: aucune paire bipolaire construite")
        return

    # 6) filtrage des signaux monopolar puis bipolarisation
    data_filt = apply_filters(
        data,
        sfreq=sfreq,
        do_notch=cfg.do_notch,
        notch_freqs=cfg.notch_freqs,
        notch_q=cfg.notch_q,
        do_highpass=cfg.do_highpass,
        highpass_hz=cfg.highpass_hz,
    )

    data_bp, bp_names = make_bipolar_data(data_filt, raw_ch_names, bipolar_pairs)
    if data_bp.size == 0 or len(bp_names) == 0:
        log(f"[WARN] {session}: pas de données bipolaires exploitables")
        return

    # 7) epochs pre/post
    pre_epochs, post_epochs, pre_times, post_times = extract_pre_post_epochs(
        data_bp=data_bp,
        sfreq=sfreq,
        trials_df=trials_df,
        pre_length=cfg.pre_length,
        post_length=cfg.post_length,
    )

    if pre_epochs.shape[0] == 0 or post_epochs.shape[0] == 0:
        log(f"[WARN] {session}: aucun epoch extrait")
        return

    # On s'aligne sur le nombre réellement extrait
    n_ok = min(pre_epochs.shape[0], post_epochs.shape[0], len(trials_df))
    pre_epochs = pre_epochs[:n_ok]
    post_epochs = post_epochs[:n_ok]
    trials_df = trials_df.iloc[:n_ok].reset_index(drop=True)

    # 8) fréquences
    freqs = make_freqs(cfg.fmin, cfg.fmax, cfg.n_freqs, cfg.freq_scale)

     # décimation des vecteurs temps
    pre_times_decim = pre_times[::cfg.decim]
    post_times_decim = post_times[::cfg.decim]

    # dossier session
    session_out = out_dir / session
    ensure_dir(session_out)

    # on sauvegarde déjà la table et les métadonnées de base
    trials_df.to_csv(session_out / f"{session}_trial_table.csv", index=False)
    np.save(session_out / f"{session}_freqs.npy", freqs)
    np.save(session_out / f"{session}_pre_times.npy", pre_times_decim)
    np.save(session_out / f"{session}_post_times.npy", post_times_decim)

    meta = {
        "session": session,
        "config": asdict(cfg),
        "n_trials": int(len(trials_df)),
        "n_raw_channels": int(len(raw_ch_names)),
        "n_bad_channels": int(len(bad_channels)),
        "n_bipolar_channels": int(len(bp_names)),
        "raw_ch_names": raw_ch_names,
        "bad_channels": bad_channels,
        "bipolar_names": bp_names,
    }
    with open(session_out / f"{session}_metadata.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    # 9) baseline globale optionnelle si demandée
    power_global_base_all = None
    if cfg.baseline_mode == "global_pre_first_stim":
        first_stim_t = float(trials_df["t_start"].min())
        base_seg, _ = build_global_baseline_segment(data_bp, sfreq, first_stim_t)

    # 10) calcul Morlet canal par canal
    for ch_idx, ch_name in enumerate(bp_names):
        log(f"[INFO] {session}: Morlet canal {ch_idx+1}/{len(bp_names)} -> {ch_name}")

        # shape (n_trials, 1, n_times)
        pre_ep_ch = pre_epochs[:, ch_idx:ch_idx+1, :]
        post_ep_ch = post_epochs[:, ch_idx:ch_idx+1, :]

        power_pre_ch = compute_morlet_power(
            epochs=pre_ep_ch,
            sfreq=sfreq,
            freqs=freqs,
            n_cycles=cfg.n_cycles,
            decim=cfg.decim,
            n_jobs=cfg.n_jobs,
        )  # (n_trials, 1, n_freqs, n_times_pre_decim)

        power_post_ch = compute_morlet_power(
            epochs=post_ep_ch,
            sfreq=sfreq,
            freqs=freqs,
            n_cycles=cfg.n_cycles,
            decim=cfg.decim,
            n_jobs=cfg.n_jobs,
        )  # (n_trials, 1, n_freqs, n_times_post_decim)

        if cfg.baseline_mode == "trial_pre":
            power_post_norm_ch, baseline_values_ch = baseline_normalize_trial_pre(
                power_pre=power_pre_ch,
                power_post=power_post_ch,
                metric=cfg.baseline_metric,
                stat=cfg.baseline_stat,
                eps=cfg.eps,
            )

        elif cfg.baseline_mode == "global_pre_first_stim":
            base_seg_ch = base_seg[ch_idx:ch_idx+1, :]  # (1, n_times)
            power_global_base_ch = compute_morlet_power(
                epochs=base_seg_ch[None, :, :],   # (1, 1, n_times)
                sfreq=sfreq,
                freqs=freqs,
                n_cycles=cfg.n_cycles,
                decim=cfg.decim,
                n_jobs=cfg.n_jobs,
            )[0]  # (1, n_freqs, n_times_base_decim)

            power_post_norm_ch, baseline_values_ch = baseline_normalize_global(
                power_post=power_post_ch,
                power_global_base=power_global_base_ch,
                metric=cfg.baseline_metric,
                stat=cfg.baseline_stat,
                eps=cfg.eps,
            )
        else:
            raise ValueError(f"baseline_mode inconnu: {cfg.baseline_mode}")

        # squeeze canal unique
        power_pre_ch = power_pre_ch[:, 0, :, :]
        power_post_ch = power_post_ch[:, 0, :, :]
        power_post_norm_ch = power_post_norm_ch[:, 0, :, :]
        baseline_values_ch = baseline_values_ch[:, 0, :] if baseline_values_ch.ndim == 3 else baseline_values_ch[0, :, :]

        # sauvegarde par canal
        safe_name = re.sub(r"[^A-Za-z0-9_\-]", "_", ch_name)
        np.save(session_out / f"{session}_power_pre_{safe_name}.npy", power_pre_ch)
        np.save(session_out / f"{session}_power_post_{safe_name}.npy", power_post_ch)
        np.save(session_out / f"{session}_power_post_norm_{safe_name}.npy", power_post_norm_ch)
        np.save(session_out / f"{session}_baseline_values_{safe_name}.npy", baseline_values_ch)

        # libération mémoire
        del pre_ep_ch, post_ep_ch
        del power_pre_ch, power_post_ch, power_post_norm_ch, baseline_values_ch

    
    
    
    
    
    
    
    
    
    
    
    
    
    
    
    # 9) Morlet pre/post
    log(f"[INFO] {session}: Morlet PRE  shape={pre_epochs.shape}")
    power_pre = compute_morlet_power(
        epochs=pre_epochs,
        sfreq=sfreq,
        freqs=freqs,
        n_cycles=cfg.n_cycles,
        decim=cfg.decim,
        n_jobs=1,
    )

    log(f"[INFO] {session}: Morlet POST shape={post_epochs.shape}")
    power_post = compute_morlet_power(
        epochs=post_epochs,
        sfreq=sfreq,
        freqs=freqs,
        n_cycles=cfg.n_cycles,
        decim=cfg.decim,
        n_jobs=1,
    )

    # décimation du vecteur temps
    pre_times_decim = pre_times[::cfg.decim]
    post_times_decim = post_times[::cfg.decim]

    # 10) baseline
    power_global_base = None

    if cfg.baseline_mode == "trial_pre":
        power_post_norm, baseline_values = baseline_normalize_trial_pre(
            power_pre=power_pre,
            power_post=power_post,
            metric=cfg.baseline_metric,
            stat=cfg.baseline_stat,
            eps=cfg.eps,
        )

    elif cfg.baseline_mode == "global_pre_first_stim":
        if not cfg.allow_global_baseline:
            raise ValueError("baseline_mode='global_pre_first_stim' mais allow_global_baseline=False")

        first_stim_t = float(trials_df["t_start"].min())
        base_seg, _ = build_global_baseline_segment(data_bp, sfreq, first_stim_t)
        # transforme en pseudo-epochs pour Morlet array: (1, n_channels, n_times)
        power_global_base = compute_morlet_power(
            epochs=base_seg[None, :, :],
            sfreq=sfreq,
            freqs=freqs,
            n_cycles=cfg.n_cycles,
            decim=cfg.decim,
            n_jobs=1,
        )[0]  # (n_channels, n_freqs, n_times)

        power_post_norm, baseline_values = baseline_normalize_global(
            power_post=power_post,
            power_global_base=power_global_base,
            metric=cfg.baseline_metric,
            stat=cfg.baseline_stat,
            eps=cfg.eps,
        )
    else:
        raise ValueError(f"baseline_mode inconnu: {cfg.baseline_mode}")

    # 11) sauvegarde
    session_out = out_dir / session
    save_session_outputs(
        session_out=session_out,
        session=session,
        config=cfg,
        trials_df=trials_df,
        raw_ch_names=raw_ch_names,
        bad_channels=bad_channels,
        bp_names=bp_names,
        freqs=freqs,
        pre_times=pre_times_decim,
        post_times=post_times_decim,
        power_pre=power_pre,
        power_post=power_post,
        power_post_norm=power_post_norm,
        baseline_values=baseline_values,
        power_global_base=power_global_base,
    )

    log(f"[OK] {session}: résultats sauvegardés dans {session_out}")


# ---------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------

def main():
    root_dir = Path(CFG.root_dir)
    out_dir = Path(CFG.output_dir)
    ensure_dir(out_dir)

    bad_df = load_bad_channels_table(root_dir)
    sessions = list_trc_sessions(root_dir)

    if len(sessions) == 0:
        raise RuntimeError(f"Aucun fichier TRC trouvé dans {root_dir}")

    log(f"{len(sessions)} sessions TRC trouvées")

    errors = []

    for session in sessions:
        try:
            process_one_session(
                session=session,
                root_dir=root_dir,
                out_dir=out_dir,
                bad_df=bad_df,
                cfg=CFG,
            )
        except Exception as exc:
            errors.append((session, repr(exc)))
            log(f"[ERROR] {session}: {exc}")

    # résumé
    summary = {
        "n_sessions": len(sessions),
        "n_errors": len(errors),
        "errors": errors,
        "config": asdict(CFG),
    }
    with open(out_dir / "run_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    log("\n=== FIN ===")
    if errors:
        log("Erreurs rencontrées:")
        for s, e in errors:
            log(f" - {s}: {e}")


if __name__ == "__main__":
    main()

2 sessions TRC trouvées

=== Session P101_DC54_stim2 ===


[INFO] P101_DC54_stim2.TRC: data shape avant RawArray = (133, 6526144)
[INFO] P101_DC54_stim2.TRC: sfreq = 2048.0
[INFO] P101_DC54_stim2.TRC: unités canal détectées = ['%', 'bpm', 'dimentionless', 'μV']
[INFO] P101_DC54_stim2.TRC: RawArray créé, shape = (133, 6526144)
[INFO] P101_DC54_stim2: Morlet PRE  shape=(42, 89, 20480)
